# Forge on your phone

Use this notebook with an eligible Colab GPU runtime. Select **Runtime → Change runtime type → L4 or A100** before starting; the complete model set requires native BF16 and does not support T4. Runtime availability and memory vary. See the [Colab usage rules](https://research.google.com/colaboratory/faq.html).

Save a copy of this notebook in Drive. Forge, its Python environments, models, settings and generated images stay in **MyDrive/ForgeColab**. The added Forge controls are in English.

**Daily use:** run **1. Connect Drive**, leave the action below on **Start Forge**, and run **2. Run one action**. Open the printed Forge link on your phone and sign in as **forge** with your session password. Keep that cell running while using Forge.

**First use:** after connecting Drive, run one action at a time in this order: **Install environments → Download models → Create Qwen NF4 → Create Qwen INT8 → Start Forge**. Installation and downloads are substantial; wait for each action to finish before changing the menu. Choose an adequately sized GPU for quantization. Existing verified conversions are reused.

**Run all executes only the single selected action.** It does not run all setup stages or start background copies. Use one running copy of this notebook per runtime.


In [ ]:
#@title 1. Connect Drive
from google.colab import drive
drive.mount("/content/drive")


Choose one action and press its play button. **Start Forge** reuses the saved installation; the other actions are for initial setup or a failed setup retry.

Downloads prefer matching Civitai files and use fixed official sources for public files missing there. WAI Illustrious SDXL v150 replaces Novsw; the three private LoRAs are excluded. The Civitai key is requested through hidden input and passed only to the downloader process. Do not paste credentials into the code or share notebook outputs containing your Forge password.


In [ ]:
#@title 2. Run one action
action = "Start Forge" #@param ["Start Forge", "Install environments", "Download models", "Create Qwen NF4", "Create Qwen INT8"]

from pathlib import Path
import getpass
import os
import runpy
import subprocess
import sys

storage = Path("/content/drive/MyDrive/ForgeColab")
repository_url = "https://github.com/pepperedmutton/stable-diffusion-webui-forge.git"


def forge_checkout(create=False):
    if sys.platform != "linux" or not os.path.ismount("/content/drive"):
        raise RuntimeError("Connect Google Drive in a Colab GPU runtime first.")
    repo = storage / "forge"
    if repo.is_symlink():
        raise RuntimeError("The Forge checkout must be a real directory on Drive.")
    if not repo.exists():
        if not create:
            raise RuntimeError("Choose Install environments before using this action.")
        storage.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--branch", "main", "--single-branch", repository_url, str(repo)], check=True)
    if not (repo / ".git").is_dir():
        raise RuntimeError("The existing forge directory is not a Git clone; it was preserved.")
    origin = subprocess.check_output(["git", "-C", str(repo), "remote", "get-url", "origin"], text=True).strip()
    if origin.rstrip("/").removesuffix(".git") != repository_url.removesuffix(".git"):
        raise RuntimeError("This checkout belongs to another repository; it was preserved.")
    return repo


def run_launcher(repo, *options):
    # Run in the notebook process so Colab can display the hidden password input.
    previous_arguments = sys.argv
    try:
        sys.argv = [str(repo / "colab/start.py"), "--drive-root", str(storage), *options]
        runpy.run_path(sys.argv[0], run_name="__main__")
    finally:
        sys.argv = previous_arguments


def run_action(selected):
    allowed = {"Start Forge", "Install environments", "Download models", "Create Qwen NF4", "Create Qwen INT8"}
    if selected not in allowed:
        raise ValueError("Choose an action from the menu.")
    repo = forge_checkout(create=selected == "Install environments")
    if selected == "Install environments":
        run_launcher(repo, "--install-only")
    elif selected == "Download models":
        key = getpass.getpass("Civitai API key: ").strip()
        if not key:
            raise ValueError("A Civitai API key is required for the complete collection.")
        environment = dict(os.environ, CIVITAI_API_KEY=key)
        try:
            subprocess.run([sys.executable, str(repo / "colab/download_models.py"), "--drive-root", str(storage), "--download"], env=environment, check=True)
        finally:
            environment.pop("CIVITAI_API_KEY", None)
            key = None
    elif selected in {"Create Qwen NF4", "Create Qwen INT8"}:
        precision = "nf4" if selected == "Create Qwen NF4" else "int8"
        python = repo / "runtimes/qwen-image-2.1/bin/python"
        if not python.is_file():
            raise RuntimeError("Install environments before creating Qwen variants.")
        subprocess.run([
            str(python), str(repo / "scripts/quantize_qwen21.py"),
            "--precision", precision, "--source", str(storage / "models/diffusers/Qwen-Image-2.1"),
            "--output", str(storage / ("models/diffusers/Qwen-Image-2.1-" + precision.upper())),
        ], cwd=repo, env=dict(os.environ, PYTHONDONTWRITEBYTECODE="1"), check=True)
    else:
        if not (repo / ".colab/installed.json").is_file():
            raise RuntimeError("Finish Install environments before starting Forge.")
        run_launcher(repo)


run_action(action)


## While Forge is running

Parameters begin locked. Tap **Edit parameters**, then **Edit value** to adjust a number. **Apply value** commits it; **Cancel** discards that draft. Tap **Finish editing** when done. The generated share link changes between sessions.

Use the running cell's **Stop** button to shut down Forge. Disconnect and delete the Colab runtime when finished; your saved Drive files remain. Do not use runtime keepalive scripts.

If installation or quantization fails, the cell raises an error and stops. Read the error before retrying the same action. An incomplete or mismatched quantization output is preserved for review. Interrupted downloads resume when you choose **Download models** again; this verifies completed files before reuse.

The notebook does not automatically update the Git checkout, repair mismatched model files, or claim every model has passed generation testing. Drive reads can make startup slow. For repair options, model details and validation limits, see the [project guide](https://github.com/pepperedmutton/stable-diffusion-webui-forge/blob/main/README.md).
